In [1]:
"""
====================================================
Download ERA5 Data
====================================================
"""

'\n====================================================\nDownload ERA5 Data\n====================================================\n'

In [2]:
#######################
# DIRECTORIES

In [3]:
#SETTING UP DIRECOTRIES
mainDirectory = "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/"
dataDirectory=mainDirectory+"../DATA/ERA5_Data/"
import os; os.makedirs(dataDirectory, exist_ok=True)

In [4]:
#######################
# LIBRARIES, FUNCTIONS, and CLASSES

In [5]:
# IMPORT LIBRARIES
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "/Libraries/"
sys.path.append(path)

# --- Import all your function modules ---
import importlib

modules = [
    "Libraries",
]

for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [6]:
# IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "Functions_2.0/"
sys.path.append(path)


# --- Import all your function modules ---
import importlib

modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [11]:
# IMPORT CLASSES
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "Functions_2.0/Classes/"
sys.path.append(path)


# --- Import all your function modules ---
import importlib

modules = [
    "Classes_InputData_DataAnalysis",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [12]:
###########################
# DOWNLOADING DATA FUNCTIONS

In [14]:
# DOWNLOADING ERA5 (Pressure Levels)
# Code Inspired from "Download_ERA5_with_python" by github.com/joaohenry23 at https://github.com/joaohenry23/Download_ERA5_with_python

#PRELIMINARY STEPS
#(1) go to https://cds.climate.copernicus.eu/how-to-api
#(2) make file in main user directory called .cdsapirc
#(3) copy the following into file: 
#    url: https://cds.climate.copernicus.eu/api
#    key: 6d55399f-dbc5-48bd-848c-31168fc0b133 (key will be different for you based on your account)
#(4) pip install "cdsapi>=0.7.4"

import cdsapi 
def DownloadERA5_PressureLevels(variables, date, area):
    c = cdsapi.Client()
    for variable in tqdm(variables, desc="Downloading ERA5 variables"):
        print(f"Downloading {variable}", "\n")
        c.retrieve(
            "reanalysis-era5-pressure-levels",
            {
                "product_type": "reanalysis",
                "format": "netcdf",
                "variable": variable,
                # "pressure_level": ['100', '250', '500', '750', '1000'], #LOW-RES
                "pressure_level": [
                    '10', '20', '30', '50', '70', 
                    '100', '125', '150', '175', '200', '225',
                    '250', '300', '350', '400', '450', '500',
                    '550', '600', '650', '700', '750', '775',
                    '800', '825', '850', '875', '900', '925',
                    '950', '975', '1000',
                ],

                "date": date,
                "time": [f"{h:02d}:00" for h in range(24)],
                "area": area,
                "grid": [0.25, 0.25],
            },
            os.path.join(
                dataDirectory, date_folder, f"{variable}_ERA5_{date_folder}.nc"
            ),
        )

#EXAMPLE RUN
# date_string = "06-30 - 07-02 (2022)"
# date_folder = MakeDateFolder(date_string)
# date_string_converted = date_string_to_range(date_string)

# # running
# DownloadERA5_PressureLevels(variables,date_string_converted, area)

In [33]:
# DOWNLOADING ERA5 (Surface)

#PRELIMINARY STEPS
#(1) go to https://cds.climate.copernicus.eu/how-to-api
#(2) make file in main user directory called .cdsapirc
#(3) copy the following into file: 
#    url: https://cds.climate.copernicus.eu/api
#    key: 6d55399f-dbc5-48bd-848c-31168fc0b133 (key will be different for you based on your account)
#(4) pip install "cdsapi>=0.7.4"

import cdsapi 
def DownloadERA5_Surface(variables, date, area):
    c = cdsapi.Client()
    for variable in tqdm(variables, desc="Downloading ERA5 surface variables"):
        print(f"Downloading {variable}", "\n")
        c.retrieve(
            "reanalysis-era5-single-levels",
            {
                "product_type": "reanalysis",
                "format": "netcdf",
                "variable": variable,
                "date": date,  # e.g. "2022-06-08/2022-06-10"
                "time": [f"{h:02d}:00" for h in range(24)],  # every hour
                "area": area,  # [North, West, South, East]
                "grid": [0.25, 0.25],
            },
            os.path.join(
                dataDirectory, date_folder, f"{variable}_ERA5_{date_folder}.nc"
            ),
        )

#EXAMPLE RUN
# date_string = "06-30 - 07-02 (2022)"
# date_folder = MakeDateFolder(date_string)
# date_string_converted = date_string_to_range(date_string)

# # running
# DownloadERA5_Surface(variables,date_string_converted, area)

In [34]:
# DATE INFORMATION
def date_string_to_range(date_string: str) -> str:
    """
    Convert a date string like "06-30 - 07-02 (2022)"
    into ERA5 API format: "2022-06-30/to/2022-07-02".
    """
    # Extract year
    year = date_string.split("(")[1].replace(")", "").strip()

    # Extract the two parts safely
    date_part = date_string.split("(")[0].strip()  # "06-30 - 07-02"
    start, end = date_part.split(" - ")            # ["06-30", "07-02"]

    # Make full YYYY-MM-DD
    start_date = f"{year}-{start}"
    end_date   = f"{year}-{end}"

    return f"{start_date}/{end_date}"

    
def MakeDateFolder(date_string):
    date_folder = strings.DateString(date_string)
    # adding date to output folder
    subdir = os.path.join(dataDirectory, date_folder)
    os.makedirs(subdir, exist_ok=True)
    return date_folder


# COORDINATES INFORMATION
def GetCoordinates(longitude, latitude, dx_m=250e3, dy_m=250e3, grid_res=0.25):
    longitude = coordinates.DMSToDecimal(*longitude)
    latitude = coordinates.DMSToDecimal(*latitude)

    dlon = coordinates.dxTOdlon(dx_m=dx_m, lat_deg=latitude)
    dlat = coordinates.dyTOdlat(dy_m=dy_m)

    N, W, S, E = [latitude + dlat, longitude - dlon, latitude - dlat, longitude + dlon]
    print("Coords box:", [N, W, S, E])
    # Round outward to 0.25 grid
    N = math.ceil(N / grid_res) * grid_res  # round north up
    S = math.floor(S / grid_res) * grid_res  # round south down
    W = math.floor(W / grid_res) * grid_res  # round west down (more negative)
    E = math.ceil(E / grid_res) * grid_res  # round east up

    area = [N, W, S, E]
    print("Rounded box:", area)
    return area


# VARIABLES INFORMATION
def GetVariableNames_PressureLevels():
    variables = [
        "u_component_of_wind",
        "v_component_of_wind",
        "vertical_velocity",
        "divergence",
        "vorticity",
        "temperature",
        "specific_humidity",
        "specific_cloud_liquid_water_content",
        "specific_cloud_ice_water_content",
        "specific_rain_water_content",
        "relative_humidity",
        "geopotential",
    ]
    return variables

def GetVariableNames_Surface():
    variables = [
        "convective_available_potential_energy",
    ]
    return variables

In [35]:
###########################
# DOWNLOADING TRACER DATA
#resolution: 72*32*20*23 = 1059840 grid-points

In [36]:
# coorindates information
# GETTING BOUNDING BOX centered at Houston, TX Mobile Facility (TRACER) Facility S2 ==> CSAP (C-Band Scanning ARM Precipitation Radar)
# 29°31'55"N, 95°17'2"W
longitude = (95, 17, 2, "W")
latitude = (29, 31, 55, "N")
area = GetCoordinates(longitude, latitude)
variables_PressureLevels = GetVariableNames_PressureLevels()
variables_Surface = GetVariableNames_Surface()

Coords box: [31.78024845924127, -97.86790574593715, 27.28364042964762, -92.69987203184061]
Rounded box: [32.0, -98.0, 27.25, -92.5]


In [37]:
###########################
# DATE ONE (BORING CASE)

In [38]:
# INFORMATION
# date information
date_string = "06-08 - 06-10 (2022)"
date_folder = MakeDateFolder(date_string)
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5_PressureLevels(variables_PressureLevels,date_string_converted, area)
DownloadERA5_Surface(variables_Surface,date_string_converted, area)

2025-09-15 18:03:28,329 INFO [2025-09-03T00:00:00] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-09-15 18:03:28,330 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-15 18:03:29,029 INFO Request ID is d9ec1965-d769-4c85-a741-643ea182012d
2025-09-15 18:03:29,195 INFO status has been updated to accepted
2025-09-15 18:03:38,340 INFO status has been updated to running
2025-09-15 18:04:02,952 INFO status has been updated to successful


42f46fc5af2a9efa161a2256f8cb42e7.nc:   0%|          | 0.00/89.1k [00:00<?, ?B/s]

In [ ]:
###########################
# DATE TWO (RAINY CASE)

In [39]:
# INFORMATION
# date information
date_string = "06-30 - 07-02 (2022)"
date_folder = MakeDateFolder(date_string)
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5_PressureLevels(variables_PressureLevels,date_string_converted, area)
DownloadERA5_Surface(variables_Surface,date_string_converted, area)

2025-09-15 18:04:05,918 INFO [2025-09-03T00:00:00] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-09-15 18:04:05,919 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-15 18:04:06,838 INFO Request ID is c9854371-d3c0-4d46-825d-e728f092d7c5
2025-09-15 18:04:06,993 INFO status has been updated to accepted
2025-09-15 18:04:21,140 INFO status has been updated to running
2025-09-15 18:04:40,605 INFO status has been updated to successful


4b4e983ff8f42dd152b0eeeb92ce6709.nc:   0%|          | 0.00/91.1k [00:00<?, ?B/s]

In [ ]:
###########################
# DATE THREE (INTERESTING CASE)

In [40]:
# INFORMATION
# date information
date_string = "08-11 - 08-13 (2022)"
date_folder = MakeDateFolder(date_string)
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5_PressureLevels(variables_PressureLevels,date_string_converted, area)
DownloadERA5_Surface(variables_Surface,date_string_converted, area)

2025-09-15 18:04:43,420 INFO [2025-09-03T00:00:00] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-09-15 18:04:43,421 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-15 18:04:44,090 INFO Request ID is 0f9c0531-e2fa-4dc1-bc25-7cf958ed483b
2025-09-15 18:04:44,900 INFO status has been updated to accepted
2025-09-15 18:04:58,939 INFO status has been updated to running
2025-09-15 18:05:18,312 INFO status has been updated to successful


c342b5fd0dbcc4c56f479df6eb3d90a2.nc:   0%|          | 0.00/89.2k [00:00<?, ?B/s]